# Nädal 7: Python ja pandas — RFM kliendisegmenteerimine

## Äriline eesmärk

UrbanStyle’i tootejuht Marko Saar soovib teada:

- kes on ettevõtte kõige väärtuslikumad kliendid;
- millised kliendid on kadumisohus;
- kuidas jaguneb kliendibaas ostukäitumise alusel;
- millistele kliendisegmentidele tuleks turundustegevustes esimesena keskenduda.

Analüüsi eesmärk on laadida UrbanStyle’i andmed Supabase’ist pandas
DataFrame’idesse, kontrollida ja töödelda andmeid ning luua RFM
(Recency, Frequency, Monetary) kliendisegmendid.


## Minu roll, töö ulatus ja tasemed

Selle nädala grupitöös on minu ametlik vastutus:

**Roll C — RFM kliendisegmenteerimine**

Minu grupitöö põhiosa hõlmab:

- Recency, Frequency ja Monetary mõõdikute arvutamist;
- R-, F- ja M-skooride määramist;
- RFM-koondskoori arvutamist;
- kliendisegmentide loomist;
- segmentide tulemuste kontrollimist ja tõlgendamist.

Käesolev notebook on minu individuaalne portfoolioartefakt. Et
analüüs oleks iseseisvalt käivitatav ja kogu töövoog mulle õppimise
eesmärgil arusaadav, olen isiklikult läbi teinud ka Roll A, Roll B ja
Roll D etapid.

Need toetavad osad ei olnud minu ametlikud vastutused grupitöös.
Need dokumenteerivad minu iseseisvalt korratud terviklikku
andmetöötluse töövoogu.

| Notebook'i jaotis | Grupitöö funktsioon | Töö liik |
|---|---|---|
| 1. Töökeskkond ja teegid | Tehniline ettevalmistus | Isiklik töö |
| 2.–4. Andmete laadimine, uurimine ja ühendamine | Roll A — Data Loading | Baastase, isiklikult korratud |
| 5. Andmete puhastamine | Roll B — Data Cleaning | Baastase, isiklikult korratud |
| 6.–9. RFM-arvutused, skoorid, segmendid ja kontroll | Roll C — RFM Analysis | Minu ametlik grupitöö roll |
| 10. Visualiseerimine | Roll D — Visualization | Baastase, isiklikult korratud |
| 11.–12. Ärisüntees, piirangud ja dokumentatsioon | QA ja ärisüntees | Isiklik laiendus |

Notebook sisaldab täielikku baastaseme töövoogu ning täiendavaid
valideerimis- ja dokumenteerimissamme. Ametliku edasijõudnute osa
täielikuks läbimiseks tuleks eraldi lisada näiteks kaalutud RFM-mudel,
detailsemad segmendid või tulemuste eksport.

## Kasutatud andmed

Analüüs kasutab järgmisi Supabase’i tabeleid:

- `sales` — müügitehingud;
- `customers` — klientide andmed;
- `products` — toodete andmed.

RFM-arvutuste põhiallikad on `sales` ja `customers`. `products`
tabelit kasutatakse koolituse näiteülesannetes ja müügiandmete
täiendavaks tõlgendamiseks.

## Töövoog

1. Töökeskkonna ja teekide kontroll
2. Andmete laadimine Supabase’ist
3. DataFrame’ide esmane uurimine
4. Tabelite ühendamine
5. Andmekvaliteedi kontroll ja puhastamine
6. RFM alusandmestiku ettevalmistamine
7. RFM mõõdikute arvutamine
8. RFM skooride ja segmentide loomine
9. Tulemuste kontroll
10. Visualiseerimine
11. Ärilised järeldused ja soovitused
12. Piirangud ja eeldused

## 1. Töökeskkond ja teegid

Selles etapis kontrollin, et notebook kasutaks õiget Pythoni
virtuaalkeskkonda ja et kõik vajalikud teegid oleksid kättesaadavad.

Kasutatavad põhiteegid:

- `pandas` — tabelandmete töötlemine;
- `supabase` — andmete laadimine Supabase’i API kaudu;
- `python-dotenv` — ühendusandmete turvaline laadimine `.env` failist;
- `plotly` — interaktiivsete visualiseeringute loomine.

Keskkonna kontroll aitab vältida olukorda, kus notebook töötab teise
Pythoni versiooni või puuduva teegiga.

In [1]:
# Põhiteegid
import os
import sys
from pathlib import Path
import importlib.metadata as metadata

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from supabase import create_client

In [2]:
print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("plotly:", metadata.version("plotly"))
print("supabase:", metadata.version("supabase"))
print("python-dotenv:", metadata.version("python-dotenv"))

Python: 3.13.12
Python executable: c:\Users\Helen\data-analysis-course\venv\Scripts\python.exe
pandas: 3.0.3
plotly: 6.8.0
supabase: 2.31.0
python-dotenv: 1.2.2


In [3]:
def find_repo_root(start_path: Path) -> Path:
    """Leiab lähima ülemkausta, milles asub .git kataloog."""
    start_path = start_path.resolve()

    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder

    raise FileNotFoundError(
        "Git-repo juurkausta ei leitud. "
        "Kontrolli, et notebook on avatud daca-portfolio projektis."
    )


repo_root = find_repo_root(Path.cwd())
env_path = repo_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(f".env faili ei leitud: {env_path}")

env_loaded = load_dotenv(
    dotenv_path=env_path,
    override=True,
)

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

print("Töökaust:", Path.cwd())
print("Repo juurkaust:", repo_root)
print(".env fail:", env_path)
print(".env laaditud:", env_loaded)
print("SUPABASE_URL olemas:", bool(supabase_url))
print("SUPABASE_KEY olemas:", bool(supabase_key))

Töökaust: c:\Users\Helen\data-analysis-course\daca-portfolio\week-7
Repo juurkaust: C:\Users\Helen\data-analysis-course\daca-portfolio
.env fail: C:\Users\Helen\data-analysis-course\daca-portfolio\.env
.env laaditud: True
SUPABASE_URL olemas: True
SUPABASE_KEY olemas: True


In [4]:
if not supabase_url or not supabase_key:
    raise RuntimeError(
        "SUPABASE_URL või SUPABASE_KEY puudub."
    )

supabase = create_client(
    supabase_url,
    supabase_key,
)

print("Supabase klient loodud: OK")


Supabase klient loodud: OK


## 2. Andmete laadimine Supabase’ist

Andmed laetakse Supabase’i tabelitest Pythonisse ja teisendatakse
pandas DataFrame’ideks.

Kuna tabelites võib olla rohkem kui 1000 rida, kasutatakse andmete
laadimiseks lehekülgede kaupa töötavat funktsiooni. Funktsioon pärib
andmeid 1000 rea kaupa, kuni kogu tabel on laaditud.

Laaditavad DataFrame’id:

- `df_sales`;
- `df_customers`;
- `df_products`.

See etapp vastab grupitöö Roll A töövoole. Isiklikus analüüsis teen
selle etapi ise, et RFM-arvutuste alusandmed oleksid enne grupitööd
olemas ja kontrollitavad.

### Andmete laadimise funktsioon

Loon korduvkasutatava `get_data()` funktsiooni, mis loeb valitud
Supabase’i tabeli kõik read 1000 rea kaupa ja tagastab need pandas
DataFrame’ina.

Funktsioon ei muuda Supabase’i andmeid ega loo andmebaasi uut tabelit.

In [5]:
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase
            .table(table_name)
            .select("*")
            .range(
                page * page_size,
                (page + 1) * page_size - 1
            )
            .execute()
        )

        data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(data)

### Analüüsiks vajalike tabelite laadimine

Kasutan `get_data()` funktsiooni, et laadida Supabase’ist kolm tabelit
eraldi pandas DataFrame’idesse:

- `sales` → `df_sales`;
- `customers` → `df_customers`;
- `products` → `df_products`.

In [6]:
df_sales = get_data("sales")
df_customers = get_data("customers")
df_products = get_data("products")

### Laaditud DataFrame’ide mahu kontroll

Kontrollin iga DataFrame’i ridade ja veergude arvu, et veenduda,
et kõik Supabase’i tabeliread laaditi täielikult.

`shape` tagastab DataFrame’i mõõtmed kujul:

`(ridade arv, veergude arv)`

In [7]:
print("sales:", df_sales.shape)
print("customers:", df_customers.shape)
print("products:", df_products.shape)

sales: (10118, 12)
customers: (3150, 9)
products: (362, 9)


## 3. DataFrame’ide esmane uurimine

Enne tabelite ühendamist ja andmete puhastamist uurin iga laaditud
DataFrame’i eraldi.

Kontrollin:

- ridade ja veergude arvu;
- veergude nimesid;
- esimesi ridu;
- veergude andmetüüpe;
- mittetühjade väärtuste arvu;
- arvuliste veergude statistilist jaotust;
- puuduvaid väärtusi;
- täielikke duplikaatridu.

Selles etapis andmeid ei muudeta. Eesmärk on mõista andmestiku
struktuuri ja tuvastada kohad, mida tuleb enne RFM-analüüsi täiendavalt
kontrollida.

### 3.1. Müügiandmete esmane uurimine

Uurin `df_sales` DataFrame’i struktuuri, sisu, andmetüüpe ja
andmekvaliteedi esmaseid näitajaid.

In [8]:
df_sales.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [9]:
print("DataFrame'i mõõtmed:", df_sales.shape)
print("Ridade arv:", df_sales.shape[0])
print("Veergude arv:", df_sales.shape[1])

print("Puuduvad sale_date väärtused:", df_sales["sale_date"].isna().sum())
print("Puuduvad customer_id väärtused:", df_sales["customer_id"].isna().sum())

print("\nVeerud:")
print(df_sales.columns.tolist())

DataFrame'i mõõtmed: (10118, 12)
Ridade arv: 10118
Veergude arv: 12
Puuduvad sale_date väärtused: 0
Puuduvad customer_id väärtused: 988

Veerud:
['id', 'sale_id', 'invoice_id', 'sale_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'channel', 'store_location', 'payment_method']


In [10]:
print("Müügiandmete veergude andmetüübid:")
print(df_sales.dtypes)

Müügiandmete veergude andmetüübid:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object


In [11]:
df_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10118 entries, 0 to 10117
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              10118 non-null  int64  
 1   sale_id         10118 non-null  int64  
 2   invoice_id      10118 non-null  str    
 3   sale_date       10118 non-null  str    
 4   customer_id     9130 non-null   float64
 5   product_id      10118 non-null  int64  
 6   quantity        10118 non-null  int64  
 7   unit_price      10118 non-null  float64
 8   total_price     10118 non-null  float64
 9   channel         10118 non-null  str    
 10  store_location  6656 non-null   str    
 11  payment_method  10118 non-null  str    
dtypes: float64(3), int64(4), str(5)
memory usage: 1.4 MB


In [12]:
df_sales.describe().T

,count,mean,std,min,25%,50%,75%,max
id,10118.0,5059.500000,2920.959346,1.00,2530.25,5059.50,7588.75,10118.00
sale_id,10118.0,5059.500000,2920.959346,1.00,2530.25,5059.50,7588.75,10118.00
customer_id,9130.0,3505.620591,842.194620,2001.00,2802.25,3556.00,4206.00,5000.00
product_id,10118.0,1175.065033,101.152004,1001.00,1086.00,1176.00,1261.00,1350.00
quantity,10118.0,1.816466,1.040837,1.00,1.00,1.00,2.00,5.00
unit_price,10118.0,164.468984,93.924072,12.83,86.14,150.11,230.06,512.37
total_price,10118.0,287.525003,273.085581,-1405.32,116.98,219.24,374.54,2170.40


In [13]:
sales_missing = (
    df_sales
    .isna()
    .sum()
    .sort_values(ascending=False)
)

sales_missing

store_location    3462
customer_id        988
id                   0
sale_id              0
sale_date            0
invoice_id           0
quantity             0
product_id           0
unit_price           0
total_price          0
channel              0
payment_method       0
dtype: int64

In [14]:
sales_duplicate_rows = df_sales.duplicated().sum()

print(
    "Täielikult dubleeritud müügiridu:",
    sales_duplicate_rows
)

Täielikult dubleeritud müügiridu: 0


In [15]:
sale_date_test = pd.to_datetime(
    df_sales["sale_date"],
    errors="coerce"
)

print(
    "Kuupäevad, mida pandas ei suutnud teisendada:",
    sale_date_test.isna().sum()
    - df_sales["sale_date"].isna().sum()
)

Kuupäevad, mida pandas ei suutnud teisendada: 0


#### Esmased tähelepanekud

- `df_sales` sisaldab 10 118 rida ja 12 veergu.
- `sale_date` laaditi Supabase’i API kaudu tekstina (`str`), kuigi
  väärtused järgivad ISO-kuupäeva vormingut.
- `sale_date` veerus puuduvad väärtused puuduvad ning kõik väärtused
  olid pandas `datetime` tüübiks teisendatavad.
- `total_price` väikseim väärtus on –1 405,32 eurot. Negatiivsed
  müügisummad võivad tähistada tagastusi või andmevigu ning nende
  käsitlus tuleb enne RFM-arvutust eraldi otsustada.
- `store_location` puudub 3 462 müügireal. Enne puuduvate väärtuste
  käsitlemist kontrollitakse, kas need read kuuluvad veebikanalisse,
  mille puhul füüsilise kaupluse asukoha puudumine on ootuspärane.
- `customer_id` laaditi tüübina `float64`, sest 988 müügireal puudub
  kliendi tunnus. Seetõttu kuvatakse olemasolevad ID-d kujul `2588.0`.
- Puuduva `customer_id`-ga müügiridu ei saa konkreetse kliendi
  RFM-analüüsi kaasata, kuid neid ei kustutata algsest
  müügiandmestikust.
- Andmetüüpe selles uurimisetapis veel ei muudeta. Teisendused tehakse
  andmete puhastamise etapis DataFrame’i koopias.

### 3.2. Kliendiandmete esmane uurimine

Uurin `df_customers` DataFrame’i struktuuri, sisu, andmetüüpe ja
andmekvaliteedi esmaseid näitajaid.

In [16]:
df_customers.head()

,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004
4,2006,Aili,Must,NaN,+372 5444 0491,Pärnu,2022-01-04,NaN,1986


In [17]:
print("DataFrame'i mõõtmed:", df_customers.shape)
print("Ridade arv:", df_customers.shape[0])
print("Veergude arv:", df_customers.shape[1])

print("\nVeerud:")
print(df_customers.columns.tolist())

DataFrame'i mõõtmed: (3150, 9)
Ridade arv: 3150
Veergude arv: 9

Veerud:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'registration_date', 'loyalty_tier', 'birth_year']


In [18]:
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   customer_id        3150 non-null   int64
 1   first_name         3150 non-null   str  
 2   last_name          3150 non-null   str  
 3   email              2770 non-null   str  
 4   phone              3150 non-null   str  
 5   city               3150 non-null   str  
 6   registration_date  3150 non-null   str  
 7   loyalty_tier       1890 non-null   str  
 8   birth_year         3150 non-null   int64
dtypes: int64(2), str(7)
memory usage: 408.8 KB


In [19]:
df_customers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,3150.0,NaN,NaN,NaN,3575.5,909.471,2001.0,2788.25,3575.5,4362.75,5150.0
first_name,3150,174,Meelis,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_name,3150,80,Rosin,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,2770,2640,mihkel.rosin@yahoo.com,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phone,3150,3000,+372 5230 7791,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,3150,12,Tallinn,1238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registration_date,3150,1508,2021-02-22,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loyalty_tier,1890,3,bronze,686,NaN,NaN,NaN,NaN,NaN,NaN,NaN
birth_year,3150.0,NaN,NaN,NaN,1982.929206,13.300914,1960.0,1971.0,1984.0,1995.0,2005.0


In [20]:
customers_missing = (
    df_customers
    .isna()
    .sum()
    .sort_values(ascending=False)
)

customers_missing

loyalty_tier         1260
email                 380
customer_id             0
last_name               0
first_name              0
phone                   0
city                    0
registration_date       0
birth_year              0
dtype: int64

In [21]:
customers_duplicate_rows = df_customers.duplicated().sum()

print(
    "Täielikult dubleeritud kliendiridu:",
    customers_duplicate_rows
)

Täielikult dubleeritud kliendiridu: 0


#### Esmased tähelepanekud

- `df_customers` sisaldab 3 150 klienti ja 9 veergu.
- `customer_id` on täisarvuline ning kõigil klientidel olemas.
- `email` puudub 380 kliendil.
- `loyalty_tier` puudub 1 260 kliendil.
- Täielikult ühesuguseid kliendiridu ei leitud.
- Mittetühje e-posti väärtusi on 2 770, kuid unikaalseid e-poste
  2 640. See viitab sellele, et osa e-posti aadresse esineb rohkem
  kui ühe kliendikirje juures.
- `registration_date` on laaditud tekstina ning vajab kuupäevana
  kasutamisel teisendamist.
- RFM-arvutuseks ei ole e-post ega lojaalsustase kohustuslikud, kuid
  nende puudumine mõjutab segmentide hilisemat turunduslikku
  kasutatavust.

### 3.3. Tooteandmete esmane uurimine

Uurin `df_products` DataFrame’i struktuuri, tooteandmete
andmetüüpe, puuduvaid väärtusi ja täielikke duplikaatridu.

In [22]:
df_products.head()

,product_id,product_name,category,subcategory,supplier,cost_price,retail_price,eco_certified,created_at
0,1001,Praktiline seemisnahkne tennised,jalanõusid,tossud,Soome Tehdas OY,99.21,157.49,False,2022-05-12
1,1002,Mugav linane linane kostüüm,meeste_riided,ülikonnad,Riia Stils SIA,173.56,274.78,True,2022-05-10
2,1003,Elegantne orgaaniline kleit,laste_riided,kleidid,Rakvere Tekstiil OÜ,31.83,52.04,False,2020-02-19
3,1004,Minimalistlik puuvillane tuunika,naiste_riided,pluusid,Vilma Design OÜ,29.41,41.63,False,2022-12-26
4,1005,Mugav tweed kardigan,meeste_riided,kampsunid,Nordic Fashion Group OÜ,135.70,198.29,True,2021-06-19


In [23]:
print("DataFrame'i mõõtmed:", df_products.shape)
print("Ridade arv:", df_products.shape[0])
print("Veergude arv:", df_products.shape[1])

print("\nVeerud:")
print(df_products.columns.tolist())

DataFrame'i mõõtmed: (362, 9)
Ridade arv: 362
Veergude arv: 9

Veerud:
['product_id', 'product_name', 'category', 'subcategory', 'supplier', 'cost_price', 'retail_price', 'eco_certified', 'created_at']


In [24]:
print("Tooteandmete veergude andmetüübid:")
print(df_products.dtypes)

Tooteandmete veergude andmetüübid:
product_id         int64
product_name         str
category             str
subcategory          str
supplier             str
cost_price       float64
retail_price     float64
eco_certified     object
created_at           str
dtype: object


In [25]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     362 non-null    int64  
 1   product_name   362 non-null    str    
 2   category       362 non-null    str    
 3   subcategory    362 non-null    str    
 4   supplier       362 non-null    str    
 5   cost_price     362 non-null    float64
 6   retail_price   362 non-null    float64
 7   eco_certified  344 non-null    object 
 8   created_at     362 non-null    str    
dtypes: float64(2), int64(1), object(1), str(5)
memory usage: 52.4+ KB


In [26]:
df_products.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,362.0,NaN,NaN,NaN,1181.5,104.644637,1001.0,1091.25,1181.5,1271.75,1362.0
product_name,362,350,Luksuslik teksane polo särk,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,362,5,meeste_riided,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subcategory,362,22,püksid,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
supplier,362,15,Vilma Design OÜ,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cost_price,362.0,NaN,NaN,NaN,108.987569,64.081933,10.15,58.8425,99.355,149.2025,346.7
retail_price,362.0,NaN,NaN,NaN,163.193564,92.697726,13.53,87.315,149.55,225.0075,434.08
eco_certified,344,2,False,242,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,362,317,2024-05-10,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
products_missing = (
    df_products
    .isna()
    .sum()
    .sort_values(ascending=False)
)

products_missing

eco_certified    18
product_id        0
product_name      0
subcategory       0
category          0
supplier          0
cost_price        0
retail_price      0
created_at        0
dtype: int64

In [28]:
products_duplicate_rows = df_products.duplicated().sum()

print(
    "Täielikult dubleeritud tooteridu:",
    products_duplicate_rows
)

Täielikult dubleeritud tooteridu: 0


#### Esmased tähelepanekud

- `df_products` sisaldab 362 rida ja 9 veergu.
- Kõigil toodetel on `product_id`, tootenimi, kategooria, hinnad ja
  tarnija.
- `eco_certified` puudub 18 tootel. Puuduv väärtus vajab hiljem
  teadlikku käsitlust ega tähenda automaatselt väärtust `False`.
- Täielikult ühesuguseid tooteridu ei leitud.
- Tootenimesid on 350 unikaalset, kuigi tooteridu on 362. Sama
  tootenimi võib seega esineda erineva `product_id` all.
- `created_at` on tekstiväljana ning vajaks kuupäevana kasutamisel
  teisendamist.
- `eco_certified` on `object` tüüpi, sest veerus esinevad nii
  tõeväärtused kui ka puuduvad väärtused.

### 3.4. Võtmeväljade unikaalsuse ja seoste kontroll

Kontrollin tabelite võtmeväljade unikaalsust ning seda, kas müügitabeli
kliendi- ja tooteviited leiavad vasted põhitabelitest.

Täieliku rea duplikaadikontroll üksi ei tuvasta olukordi, kus sama
äriline tunnus, näiteks e-post või toote ID, esineb mitmes erinevas
kirjes.

In [29]:
print("sales sale_id duplikaadid:",
      df_sales["sale_id"].duplicated().sum())

print("sales invoice_id duplikaadid:",
      df_sales["invoice_id"].duplicated().sum())

print("customers customer_id duplikaadid:",
      df_customers["customer_id"].duplicated().sum())

print("products product_id duplikaadid:",
      df_products["product_id"].duplicated().sum())

sales sale_id duplikaadid: 0
sales invoice_id duplikaadid: 0
customers customer_id duplikaadid: 0
products product_id duplikaadid: 0


In [30]:
duplicate_emails = (
    df_customers
    .dropna(subset=["email"])
    .duplicated(subset=["email"], keep=False)
    .sum()
)

print("Korduva e-postiga kliendiread:", duplicate_emails)

Korduva e-postiga kliendiread: 258


In [31]:
sales_customer_ids = set(
    df_sales["customer_id"].dropna().astype("int64")
)

customer_ids = set(df_customers["customer_id"])

sales_product_ids = set(df_sales["product_id"])
product_ids = set(df_products["product_id"])

print(
    "Klienditabelis vasteta customer_id väärtused:",
    len(sales_customer_ids - customer_ids)
)

print(
    "Tootetabelis vasteta product_id väärtused:",
    len(sales_product_ids - product_ids)
)

Klienditabelis vasteta customer_id väärtused: 0
Tootetabelis vasteta product_id väärtused: 0


#### Võtmekontrollide tulemused

- `sale_id`, `invoice_id`, `customer_id` ja `product_id`
  ühendamisvõtmetes duplikaate ei leitud.
- Kõik müügitabeli olemasolevad `customer_id` väärtused leiavad
  `customers` tabelist vaste.
- Kõik müügitabeli `product_id` väärtused leiavad `products`
  tabelist vaste.
- Müügitabelis on lisaks 988 puuduva `customer_id`-ga rida, mida ei
  saa siduda konkreetse kliendiga ega kasutada kliendipõhises
  RFM-analüüsis.
- 258 kliendirida kuuluvad korduva e-posti aadressiga gruppidesse.
  Neid ei eemaldata automaatselt, sest sama e-posti kasutamine ei
  tõenda üheselt, et tegemist on sama kliendiga.
- Tabelite ühendamiseks kasutatavad ID-väljad on sobivad ning
  ühendamine ei tohiks tekitada võtmeduplikaatidest põhjustatud
  ridade paljunemist.

## 4. Tabelite ühendamine

Müügiandmed ühendatakse kliendi- ja tooteandmetega pandas
`merge()` meetodi abil.

Ühendamisvõtmed:

- `sales.customer_id` → `customers.customer_id`;
- `sales.product_id` → `products.product_id`.

Kasutan vasakpoolset ühendamist (`how="left"`), et säilitada kõik
10 118 müügirida. Puuduva `customer_id`-ga müügiread jäävad
andmestikku alles, kuid nende kliendiandmete väljad on pärast
ühendamist puuduvad.

Parameeter `validate="many_to_one"` kontrollib, et mitmele
müügireale vastaks kliendi- või tootetabelis maksimaalselt üks
kirje. See aitab vältida ridade ootamatut paljunemist.

### 4.1. Müügi- ja kliendiandmete ühendamine

In [32]:
df_merged = df_sales.merge(
    df_customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_match"
)

print("Müügi- ja kliendiandmete ühendamise tulemus:")
print(df_merged.shape)

print("\nKliendiühenduse staatus:")
print(df_merged["_customer_match"].value_counts())

Müügi- ja kliendiandmete ühendamise tulemus:
(10118, 21)

Kliendiühenduse staatus:
_customer_match
both          9130
left_only      988
right_only       0
Name: count, dtype: int64


### 4.2. Tooteandmete ühendamine

In [33]:
df_merged = df_merged.merge(
    df_products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="_product_match"
)

print("Kõigi kolme tabeli ühendamise tulemus:")
print(df_merged.shape)

print("\nTooteühenduse staatus:")
print(df_merged["_product_match"].value_counts())

Kõigi kolme tabeli ühendamise tulemus:
(10118, 30)

Tooteühenduse staatus:
_product_match
both          10118
left_only         0
right_only        0
Name: count, dtype: int64


### 4.3. Ridade arvu kontroll enne ja pärast ühendamist

In [34]:
print("Müügiridu enne ühendamist:", len(df_sales))
print("Ridu pärast ühendamist:", len(df_merged))
print(
    "Ridade arvu muutus:",
    len(df_merged) - len(df_sales)
)

Müügiridu enne ühendamist: 10118
Ridu pärast ühendamist: 10118
Ridade arvu muutus: 0


#### Ühendamise tulemused

- Müügi- ja kliendiandmete ühendamisel säilisid kõik 10 118
  müügirida.
- 9 130 müügirida leidsid `customers` tabelist kliendi vaste.
- 988 müügireal puudus `customer_id`, mistõttu nende kliendiandmete
  väljad jäid pärast ühendamist tühjaks.
- Kõik 10 118 müügirida leidsid `products` tabelist toote vaste.
- Ridade arv enne ja pärast ühendamist jäi samaks, mistõttu
  ühendamine ei põhjustanud ridade paljunemist.
- Ühendamise kontrollimiseks lisatud tehnilised veerud
  `_customer_match` ja `_product_match` eemaldatakse enne lõplikku
  analüüsi.

## 5. Andmekvaliteedi kontroll ja puhastamine

Loon ühendatud andmestikust eraldi töökoopia, et algne `df_merged`
säiliks muutmata kujul.

Puhastamise eesmärk on valmistada ette usaldusväärne andmestik
kliendipõhiseks RFM-analüüsiks. Kontrollin:

- täielikke duplikaatridu;
- RFM-i jaoks kriitiliste väljade puuduvaid väärtusi;
- kuupäevade teisendatavust ja vahemikku;
- null- ja negatiivseid müügisummasid;
- müügikanali ja kaupluse asukoha omavahelist loogikat.

Puuduva `customer_id`-ga read võivad olla kehtivad külalisostud.
Neid ei kustutata algsest andmestikust, kuid neid ei saa kasutada
kliendipõhises RFM-analüüsis.

### 5.1. Puhastamise töökoopia ja lähteolukord

Loon `df_merged` DataFrame’ist koopia. Kõik puhastustoimingud tehakse
koopias, et ühendatud algandmestik jääks kontrollimiseks alles.

In [35]:
df_clean = df_merged.copy()

print("Algne ühendatud andmestik:", df_merged.shape)
print("Puhastamise töökoopia:", df_clean.shape)

Algne ühendatud andmestik: (10118, 30)
Puhastamise töökoopia: (10118, 30)


#### Eemaldan ühendamise tehnilised kontrollveerud

Need kaks veergu olid vajalikud ainult merge() tulemuse kontrollimiseks:

    _customer_match
    _product_match

errors="ignore" tähendab, et kood ei anna viga ka siis, kui mõni neist veergudest on juba eemaldatud.

In [36]:
match_columns = [
    "_customer_match",
    "_product_match"
]

df_clean = df_clean.drop(
    columns=match_columns,
    errors="ignore"
)

print("Mõõtmed pärast tehniliste veergude eemaldamist:", df_clean.shape)

Mõõtmed pärast tehniliste veergude eemaldamist: (10118, 28)


### 5.2. RFM-i kriitiliste väljade kontroll

RFM-analüüs vajab iga tehingu kohta:

- kliendi tunnust (`customer_id`);
- müügikuupäeva (`sale_date`);
- tehingu rahalist väärtust (`total_price`).

Kontrollin enne puhastamist, mitmel real mõni neist väärtustest puudub.

#### Senise kontrolli põhjal puudub `customer_id` 988 real, kuid kontrollin seda ühendatud töökoopias uuesti.

In [37]:
rfm_required_columns = [
    "customer_id",
    "sale_date",
    "total_price"
]

critical_missing = (
    df_clean[rfm_required_columns]
    .isna()
    .sum()
)

print("Puuduvad väärtused RFM-i kriitilistes veergudes:")
print(critical_missing)

Puuduvad väärtused RFM-i kriitilistes veergudes:
customer_id    988
sale_date        0
total_price      0
dtype: int64


#### kontroll, kas mõnel real puudub mitu kriitilist väärtust korraga:

In [38]:
rows_with_critical_missing = (
    df_clean[rfm_required_columns]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    "Vähemalt ühe kriitilise väärtuseta ridu:",
    rows_with_critical_missing
)

Vähemalt ühe kriitilise väärtuseta ridu: 988


### 5.3. Andmetüüpide korrastamine

Teisendan `sale_date` veeru pandas kuupäevatüübiks ja
`customer_id` veeru nullable täisarvutüübiks `Int64`.

`Int64` võimaldab säilitada puuduvaid kliendi-ID väärtusi kujul
`<NA>` kuni RFM-i jaoks sobimatute ridade filtreerimiseni.

In [39]:
df_clean["sale_date"] = pd.to_datetime(
    df_clean["sale_date"],
    errors="coerce"
)

df_clean["customer_id"] = (
    pd.to_numeric(
        df_clean["customer_id"],
        errors="coerce"
    )
    .astype("Int64")
)

print("sale_date tüüp:", df_clean["sale_date"].dtype)
print("customer_id tüüp:", df_clean["customer_id"].dtype)

sale_date tüüp: datetime64[us]
customer_id tüüp: Int64


#### Kontrollin kuupäevade vahemikku

In [40]:
print("Varaseim müügikuupäev:", df_clean["sale_date"].min())
print("Hiliseim müügikuupäev:", df_clean["sale_date"].max())
print(
    "Teisendamise järel puuduvad kuupäevad:",
    df_clean["sale_date"].isna().sum()
)

Varaseim müügikuupäev: 2023-01-01 00:00:00
Hiliseim müügikuupäev: 2026-06-28 00:00:00
Teisendamise järel puuduvad kuupäevad: 0


### 5.4. Duplikaatide ja müügisummade kontroll

Kontrollin enne filtreerimist täielikke duplikaatridu ning null- ja
negatiivseid müügisummasid.

Negatiivne `total_price` võib tähistada tagastust või andmeviga.
Koolituse RFM-töövoos jäetakse RFM-arvutusest välja kõik read, mille
müügisumma ei ole positiivne.

In [41]:
print(
    "Täielikud duplikaatridad:",
    df_clean.duplicated().sum()
)

print(
    "Negatiivse total_price väärtusega read:",
    (df_clean["total_price"] < 0).sum()
)

print(
    "Nullväärtusega total_price read:",
    (df_clean["total_price"] == 0).sum()
)

print(
    "Positiivse total_price väärtusega read:",
    (df_clean["total_price"] > 0).sum()
)

Täielikud duplikaatridad: 0
Negatiivse total_price väärtusega read: 195
Nullväärtusega total_price read: 0
Positiivse total_price väärtusega read: 9923


#### negatiivsete ridade vaatamiseks

In [42]:
negative_sales = (
    df_clean.loc[
        df_clean["total_price"] < 0,
        [
            "sale_id",
            "invoice_id",
            "sale_date",
            "customer_id",
            "product_id",
            "quantity",
            "unit_price",
            "total_price",
            "channel",
            "store_location"
        ]
    ]
    .sort_values("total_price")
)

negative_sales.head(10)

,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location
4220,4221,INV-202312-00388,2023-12-05,2412,1293,4,351.33,-1405.32,pood,Tallinn
4032,4033,INV-202312-00200,2023-12-15,2844,1212,4,276.20,-1104.80,pood,Tallinn
8603,8604,INV-202411-00105,2024-11-07,4731,1171,3,351.62,-1054.86,online,NaN
4195,4196,INV-202312-00363,2023-12-23,4193,1168,3,316.04,-948.12,pood,Pärnu
7209,7210,INV-202408-00008,2024-08-03,3682,1314,4,234.91,-939.64,online,NaN
1069,1070,INV-202304-00187,2023-04-06,2609,1198,3,291.72,-875.16,pood,Tallinn
1511,1512,INV-202305-00285,2023-05-02,3970,1166,4,205.00,-820.00,online,NaN
5483,5484,INV-202404-00130,2024-04-07,4746,1185,4,203.67,-814.68,pood,Tallinn
1302,1303,INV-202305-00076,2023-05-17,2228,1350,3,270.69,-812.07,online,NaN
6993,6994,INV-202407-00303,2024-07-20,3446,1350,3,270.69,-812.07,pood,Tallinn


### 5.5. Müügikanali ja kaupluse asukoha kontroll

Kontrollin, kas puuduv `store_location` on seotud veebikanaliga.
Sellisel juhul ei ole puuduv füüsilise kaupluse asukoht tingimata
andmeviga.

In [43]:
channel_location_check = pd.crosstab(
    df_clean["channel"],
    df_clean["store_location"].fillna("Puudub"),
    margins=True
)

channel_location_check

store_location,Puudub,Pärnu,Tallinn,Tartu,All
channel,,,,,
online,3462,0,0,0,3462
pood,0,1058,3801,1797,6656
All,3462,1058,3801,1797,10118


#### Täpsem kokkuvõte

In [44]:
missing_location_by_channel = (
    df_clean.loc[df_clean["store_location"].isna()]
    .groupby("channel", dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("Puuduva store_location väärtusega read kanalite kaupa:")
print(missing_location_by_channel)

Puuduva store_location väärtusega read kanalite kaupa:
channel
online    3462
dtype: int64


### 5.6. RFM-analüüsiks sobivate tehingute filtreerimine

Loon puhastatud üldandmestikust eraldi RFM-tehingute DataFrame’i.

RFM-analüüsist jäetakse välja read, millel:

- puudub `customer_id`;
- puudub `sale_date`;
- puudub `total_price`;
- `total_price` on null või negatiivne.

Negatiivse müügisummaga read säilivad üldises `df_clean`
andmestikus, kuid neid ei kasutata koolituse RFM-arvutuses.

Kontrollin välistamise põhjuseid eraldi ja koos, sest ühel real võib
esineda mitu probleemi korraga.

#### Koostan välistamise põhjuste kontroll

In [45]:
rfm_exclusion_reasons = pd.DataFrame({
    "missing_customer_id": df_clean["customer_id"].isna(),
    "missing_sale_date": df_clean["sale_date"].isna(),
    "missing_total_price": df_clean["total_price"].isna(),
    "non_positive_total_price": df_clean["total_price"].le(0)
})

print("RFM-ist välistamise põhjused:")
print(rfm_exclusion_reasons.sum())

RFM-ist välistamise põhjused:
missing_customer_id         988
missing_sale_date             0
missing_total_price           0
non_positive_total_price    195
dtype: int64


#### Kontrollin probleemide kattumist

In [46]:
rows_with_multiple_issues = (
    rfm_exclusion_reasons
    .sum(axis=1)
    .gt(1)
    .sum()
)

print(
    "Mitme välistamise põhjusega ridu:",
    rows_with_multiple_issues
)

Mitme välistamise põhjusega ridu: 15


#### RFM-tehingute andmestiku loomine

In [47]:
rfm_exclusion_mask = rfm_exclusion_reasons.any(axis=1)

df_rfm_transactions = (
    df_clean
    .loc[~rfm_exclusion_mask]
    .copy()
)

print("Ridu enne RFM-filtreerimist:", len(df_clean))
print(
    "RFM-ist välistatud ridu:",
    rfm_exclusion_mask.sum()
)
print(
    "RFM-analüüsi sobivaid ridu:",
    len(df_rfm_transactions)
)

Ridu enne RFM-filtreerimist: 10118
RFM-ist välistatud ridu: 1168
RFM-analüüsi sobivaid ridu: 8950


#### Puhastusraport

In [48]:
rfm_cleaning_summary = pd.Series({
    "Read enne filtreerimist": len(df_clean),
    "Puuduva customer_id-ga read":
        rfm_exclusion_reasons["missing_customer_id"].sum(),
    "Puuduva sale_date-ga read":
        rfm_exclusion_reasons["missing_sale_date"].sum(),
    "Puuduva total_price-ga read":
        rfm_exclusion_reasons["missing_total_price"].sum(),
    "Null- või negatiivse total_price-ga read":
        rfm_exclusion_reasons["non_positive_total_price"].sum(),
    "Mitme probleemiga read": rows_with_multiple_issues,
    "Välistatud unikaalsed read": rfm_exclusion_mask.sum(),
    "RFM-i sobivad read": len(df_rfm_transactions)
})

rfm_cleaning_summary

Read enne filtreerimist                     10118
Puuduva customer_id-ga read                   988
Puuduva sale_date-ga read                       0
Puuduva total_price-ga read                     0
Null- või negatiivse total_price-ga read      195
Mitme probleemiga read                         15
Välistatud unikaalsed read                   1168
RFM-i sobivad read                           8950
dtype: int64

#### Lõpptulemuse kontroll

In [49]:
print(
    "Puuduvad customer_id väärtused:",
    df_rfm_transactions["customer_id"].isna().sum()
)

print(
    "Puuduvad sale_date väärtused:",
    df_rfm_transactions["sale_date"].isna().sum()
)

print(
    "Puuduvad total_price väärtused:",
    df_rfm_transactions["total_price"].isna().sum()
)

print(
    "Null- või negatiivsed total_price väärtused:",
    (df_rfm_transactions["total_price"] <= 0).sum()
)

Puuduvad customer_id väärtused: 0
Puuduvad sale_date väärtused: 0
Puuduvad total_price väärtused: 0
Null- või negatiivsed total_price väärtused: 0


#### Puhastamise kokkuvõte

- Ühendatud lähteandmestikus oli 10 118 müügirida.
- `customer_id` puudus 988 real.
- `sale_date` ja `total_price` kriitilisi puuduvaid väärtusi ei olnud.
- Negatiivse müügisummaga ridu oli 195 ning nullsummaga ridu ei olnud.
- 15 real esines korraga rohkem kui üks välistamise põhjus.
- RFM-analüüsist välistati kokku 1 168 unikaalset müügirida.
- RFM-analüüsiks jäi 8 950 sobivat tehingurida.
- Lõplikus RFM-tehingute andmestikus ei ole puuduvaid kliendi-ID-sid,
  kuupäevi, müügisummasid ega null- või negatiivseid müügisummasid.
- Puuduv `store_location` esines ainult veebikanali müükidel ning
  seda ei käsitletud andmeveana.

## 6. RFM-alusandmestiku ettevalmistamine

RFM-analüüsiks kasutan puhastatud tehinguandmestikust järgmisi välju:

- `customer_id` — klient, kelle kohta mõõdikud arvutatakse;
- `sale_id` — müügitehingu tunnus Frequency arvutamiseks;
- `sale_date` — ostukuupäev Recency arvutamiseks;
- `total_price` — tehingu väärtus Monetary arvutamiseks.

Toote-, kontakt- ja müügikanali andmeid ei ole RFM-mõõdikute
arvutamiseks otseselt vaja. Need säilivad üldises puhastatud
andmestikus ja neid saab hiljem kasutada tulemuste tõlgendamisel.

In [50]:
rfm_base = (
    df_rfm_transactions[
        [
            "customer_id",
            "sale_id",
            "sale_date",
            "total_price"
        ]
    ]
    .copy()
)

print("RFM-alusandmestiku mõõtmed:", rfm_base.shape)
print("Unikaalseid kliente:", rfm_base["customer_id"].nunique())
print("Unikaalseid müügitehinguid:", rfm_base["sale_id"].nunique())

RFM-alusandmestiku mõõtmed: (8950, 4)
Unikaalseid kliente: 2540
Unikaalseid müügitehinguid: 8950


### 6.1. Analüüsi viitekuupäev

Recency arvutamiseks on vaja määrata kuupäev, mille suhtes mõõdetakse
kliendi viimasest ostust möödunud aega.

Koolitusjuhendi näites kasutatud kuupäev `2025-02-28` ei sobi selle
andmestikuga, sest müügiandmed ulatuvad 2026. aastasse. Selle
kasutamine tekitaks hilisematele tehingutele negatiivsed Recency
väärtused.

Kasutan viitekuupäevana andmestiku viimasele müügikuupäevale
järgnevat päeva. Nii on arvutus reprodutseeritav ja kõigi klientide
Recency väärtused jäävad mittenegatiivseks.

In [51]:
analysis_date = (
    rfm_base["sale_date"]
    .max()
    .normalize()
    + pd.Timedelta(days=1)
)

print("Viimane müügikuupäev:", rfm_base["sale_date"].max())
print("RFM-analüüsi viitekuupäev:", analysis_date)

Viimane müügikuupäev: 2026-06-28 00:00:00
RFM-analüüsi viitekuupäev: 2026-06-29 00:00:00


### 6.2. RFM-alusandmestiku kontroll

Enne kliendipõhiste mõõdikute arvutamist kontrollin veel kord
alusandmestiku täielikkust, andmetüüpe ja kuupäevade sobivust.

In [52]:
print("Puuduvad väärtused:")
print(rfm_base.isna().sum())

print("\nAndmetüübid:")
print(rfm_base.dtypes)

print(
    "\nKõik müügikuupäevad on viitekuupäevast varasemad:",
    (rfm_base["sale_date"] < analysis_date).all()
)

print(
    "Kõik müügisummad on positiivsed:",
    (rfm_base["total_price"] > 0).all()
)

print(
    "sale_id duplikaadid:",
    rfm_base["sale_id"].duplicated().sum()
)

Puuduvad väärtused:
customer_id    0
sale_id        0
sale_date      0
total_price    0
dtype: int64

Andmetüübid:
customer_id             Int64
sale_id                 int64
sale_date      datetime64[us]
total_price           float64
dtype: object

Kõik müügikuupäevad on viitekuupäevast varasemad: True
Kõik müügisummad on positiivsed: True
sale_id duplikaadid: 0


#### RFM-alusandmestiku kontrolli tulemused

- RFM-alusandmestikus on 8 950 sobivat müügitehingut.
- Andmestikus on 2 540 unikaalset klienti.
- Kõik 8 950 `sale_id` väärtust on unikaalsed.
- Kliendi-ID, müügikuupäeva ega müügisumma puuduvaid väärtusi ei ole.
- Kõik müügisummad on positiivsed.
- Kõik müügikuupäevad on analüüsi viitekuupäevast varasemad.
- RFM-mõõdikute arvutamise viitekuupäev on 2026-06-29.

## 7. RFM-mõõdikute arvutamine

Arvutan iga kliendi kohta kolm RFM-mõõdikut:

- **Recency** – päevade arv analüüsi viitekuupäeva ja kliendi
  viimase ostu vahel;
- **Frequency** – kliendi unikaalsete müügitehingute arv;
- **Monetary** – kliendi positiivsete müügitehingute kogusumma.

Lisaks säilitan kontrollimiseks kliendi viimase ostu kuupäeva.

### 7.1. Tehingute koondamine klientide kaupa

Kasutan pandas `groupby()` ja `agg()` meetodeid, et koondada
tehingutaseme andmed üheks reaks iga kliendi kohta.

`as_index=False` jätab `customer_id` tavaliseks DataFrame’i veeruks,
mitte indeksi osaks.

`frequency=("sale_id", "nunique")` tähendab võta veerg 'sale_id' ja 
loenda iga kliendi unikaalsed väärtused ning nimeta uus tulemusveerg frequency.
`monetary=("total_price", "sum")`  liidab iga kliendi kõik positiivsed müügisummad.

In [53]:
rfm_metrics = (
    rfm_base
    .groupby(
        "customer_id",
        as_index=False
    )
    .agg(
        last_purchase_date=("sale_date", "max"),
        frequency=("sale_id", "nunique"),
        monetary=("total_price", "sum")
    )
)

print("Kliendipõhise koondtabeli mõõtmed:", rfm_metrics.shape)
rfm_metrics.head()

Kliendipõhise koondtabeli mõõtmed: (2540, 4)


,customer_id,last_purchase_date,frequency,monetary
0,2001,2024-11-29,2,203.92
1,2004,2024-12-19,2,1198.56
2,2005,2024-10-03,4,959.60
3,2006,2023-11-09,1,327.06
4,2007,2025-01-30,1,318.63


### 7.2. Recency arvutamine

Recency arvutatakse analüüsi viitekuupäeva ja kliendi viimase
ostukuupäeva vahena päevades.

Kuna viitekuupäev on üks päev pärast kogu andmestiku viimast
müügikuupäeva, on kõige hiljuti ostnud klientide Recency väärtus
vähemalt 1 päev.

In [54]:
rfm_metrics["recency"] = (
    analysis_date
    - rfm_metrics["last_purchase_date"]
).dt.days

rfm_metrics = rfm_metrics[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "last_purchase_date"
    ]
]

rfm_metrics.head()

,customer_id,recency,frequency,monetary,last_purchase_date
0,2001,577,2,203.92,2024-11-29
1,2004,557,2,1198.56,2024-12-19
2,2005,634,4,959.60,2024-10-03
3,2006,963,1,327.06,2023-11-09
4,2007,515,1,318.63,2025-01-30


### 7.3. RFM-mõõdikute kontroll

Kontrollin, et:

- iga klient esineb koondtabelis ühe korra;
- RFM-väärtused ei puudu;
- Recency ja Frequency on vähemalt 1;
- Monetary on positiivne;
- tehingute arv ja kogusumma säilisid koondamisel.

In [55]:
print("RFM-tabeli mõõtmed:", rfm_metrics.shape)

print(
    "Unikaalseid customer_id väärtusi:",
    rfm_metrics["customer_id"].nunique()
)

print(
    "Dubleeritud customer_id väärtusi:",
    rfm_metrics["customer_id"].duplicated().sum()
)

print("\nPuuduvad RFM-väärtused:")
print(
    rfm_metrics[
        ["recency", "frequency", "monetary"]
    ]
    .isna()
    .sum()
)

RFM-tabeli mõõtmed: (2540, 5)
Unikaalseid customer_id väärtusi: 2540
Dubleeritud customer_id väärtusi: 0

Puuduvad RFM-väärtused:
recency      0
frequency    0
monetary     0
dtype: int64


#### Kontrollin, et kõik RFM-i veerud on vähemalt 1 või positiivsed.

In [56]:
print(
    "Kõik Recency väärtused on vähemalt 1:",
    rfm_metrics["recency"].ge(1).all()
)

print(
    "Kõik Frequency väärtused on vähemalt 1:",
    rfm_metrics["frequency"].ge(1).all()
)

print(
    "Kõik Monetary väärtused on positiivsed:",
    rfm_metrics["monetary"].gt(0).all()
)

Kõik Recency väärtused on vähemalt 1: True
Kõik Frequency väärtused on vähemalt 1: True
Kõik Monetary väärtused on positiivsed: True


#### Koondamise kontroll lähteandmete vastu

In [57]:
frequency_matches = (
    rfm_metrics["frequency"].sum()
    == rfm_base["sale_id"].nunique()
)

monetary_matches = (
    round(rfm_metrics["monetary"].sum(), 2)
    == round(rfm_base["total_price"].sum(), 2)
)

print(
    "Tehingute arv säilis koondamisel:",
    frequency_matches
)

print(
    "Müügisumma säilis koondamisel:",
    monetary_matches
)

print(
    "RFM-tabeli tehingute koguarv:",
    rfm_metrics["frequency"].sum()
)

print(
    "Lähteandmestiku tehingute koguarv:",
    rfm_base["sale_id"].nunique()
)

Tehingute arv säilis koondamisel: True
Müügisumma säilis koondamisel: True
RFM-tabeli tehingute koguarv: 8950
Lähteandmestiku tehingute koguarv: 8950


### 7.4. RFM-mõõdikute statistiline ülevaade

Statistiline ülevaade aitab hinnata RFM-väärtuste jaotust ja
võimalikke äärmuslikke väärtusi enne skooride määramist.

In [58]:
rfm_metrics[
    [
        "recency",
        "frequency",
        "monetary"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
recency,2540.0,684.044094,175.547883,1.00,555.000,646.50,774.0000,1269.00
frequency,2540.0,3.523622,5.354013,1.00,2.000,3.00,4.0000,77.00
monetary,2540.0,1053.878165,1752.962707,15.09,410.175,784.35,1287.0875,27920.86


#### RFM-mõõdikute kontrolli tulemused

- RFM-tabel sisaldab 2 540 unikaalset klienti.
- Iga klient esineb tabelis ühe korra ning RFM-väärtustes puuduvad
  väärtused puuduvad.
- Tehingute arv ja kogusumma säilisid kliendipõhisel koondamisel.
- Recency mediaan on 646,5 päeva.
- Frequency mediaan on 3 tehingut ning maksimum 77 tehingut.
- Monetary mediaan on 784,35 eurot ning maksimum 27 920,86 eurot.
- Frequency ja Monetary jaotused on paremale kaldu, mis näitab
  väikese arvu väga aktiivsete ja kõrge väärtusega klientide olemasolu.
- Kvintiilipõhine skoorimine võimaldab hinnata kliente kogu
  kliendibaasi suhtes, vähendades äärmuslike väärtuste mõju
  segmentide piiridele.

## 8. RFM-skooride ja segmentide loomine

Määran igale kliendile Recency, Frequency ja Monetary skoori
vahemikus 1–5.

Skoorimise loogika:

- **Recency:** väiksem päevade arv on parem, mistõttu kõige hiljuti
  ostnud kliendid saavad skoori 5;
- **Frequency:** suurem ostude arv on parem;
- **Monetary:** suurem kogukulutus on parem.

Skoorid määratakse kvintiilide alusel, võrreldes iga klienti ülejäänud
kliendibaasiga.

Frequency väärtustes esineb palju võrdseid täisarve. Seetõttu kasutan
enne kvintiilide moodustamist `rank(method="first")` meetodit, et
`pd.qcut()` saaks kliendid viide rühma jagada.

#### Töökoopia loomine

In [59]:
rfm_scored = rfm_metrics.copy()

print("Skoorimise lähteandmestik:", rfm_scored.shape)

Skoorimise lähteandmestik: (2540, 5)


#### Määran R-, F- ja M-skoorid
Koodi loogika

`Recency` puhul:
labels=[5, 4, 3, 2, 1] annab kõige väiksematele Recency väärtustele skoori 5.

`Frequency` ja `Monetary` puhul:
labels=[1, 2, 3, 4, 5] annab kõige suurematele väärtustele skoori 5.

Juhendis on `Frequency` puhul samuti soovitatud kasutada enne `qcut()` meetodit järjestamist, sest ostude arv sisaldab palju võrdseid väärtusi.

In [60]:
rfm_scored["R_score"] = pd.qcut(
    rfm_scored["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype("int64")

rfm_scored["F_score"] = pd.qcut(
    rfm_scored["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype("int64")

rfm_scored["M_score"] = pd.qcut(
    rfm_scored["monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype("int64")

### 8.1. RFM-skooride kontroll

Kontrollin, et iga mõõdiku skoorid jäävad vahemikku 1–5 ning et kõik
viis skooritaset on andmestikus esindatud.

In [61]:
score_columns = [
    "R_score",
    "F_score",
    "M_score"
]

print("Skooride miinimumid ja maksimumid:")
print(
    rfm_scored[score_columns]
    .agg(["min", "max"])
)

for column in score_columns:
    print(f"\n{column} jaotus:")
    print(
        rfm_scored[column]
        .value_counts()
        .sort_index()
    )

Skooride miinimumid ja maksimumid:
     R_score  F_score  M_score
min        1        1        1
max        5        5        5

R_score jaotus:
R_score
1    507
2    507
3    507
4    508
5    511
Name: count, dtype: int64

F_score jaotus:
F_score
1    508
2    508
3    508
4    508
5    508
Name: count, dtype: int64

M_score jaotus:
M_score
1    510
2    506
3    508
4    508
5    508
Name: count, dtype: int64


### 8.2. RFM-koondskoori arvutamine

RFM-koondskoor saadakse kolme osaskoori liitmisel.

Väikseim võimalik tulemus on 3 ja suurim võimalik tulemus 15.
Kõrgem koondskoor näitab üldiselt hiljutisemat, sagedasemat ja
suurema väärtusega ostukäitumist.

In [62]:
rfm_scored["RFM_Score"] = (
    rfm_scored["R_score"]
    + rfm_scored["F_score"]
    + rfm_scored["M_score"]
)

print(
    "RFM-koondskoori vahemik:",
    rfm_scored["RFM_Score"].min(),
    "kuni",
    rfm_scored["RFM_Score"].max()
)

rfm_scored[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Score"
    ]
].head()

RFM-koondskoori vahemik: 3 kuni 15


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_Score
0,2001,577,2,203.92,4,1,1,6
1,2004,557,2,1198.56,4,1,4,9
2,2005,634,4,959.60,3,4,3,10
3,2006,963,1,327.06,1,1,1,3
4,2007,515,1,318.63,5,1,1,7


### 8.3. Kliendisegmentide määramine

Jaotan kliendid RFM-koondskoori alusel viide segmenti:

- **VIP Champions** — skoor 13–15;
- **Loyal** — skoor 10–12;
- **Potential** — skoor 7–9;
- **At Risk** — skoor 4–6;
- **Lost** — skoor 3.

Segment näitab kliendi ostukäitumise profiili, mitte kindlat ennustust
tema tulevase käitumise kohta.

In [63]:
def assign_rfm_segment(rfm_score):
    if rfm_score >= 13:
        return "VIP Champions"
    elif rfm_score >= 10:
        return "Loyal"
    elif rfm_score >= 7:
        return "Potential"
    elif rfm_score >= 4:
        return "At Risk"
    else:
        return "Lost"


rfm_scored["Segment"] = (
    rfm_scored["RFM_Score"]
    .apply(assign_rfm_segment)
)

rfm_scored[
    [
        "customer_id",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Score",
        "Segment"
    ]
].head(10)

,customer_id,R_score,F_score,M_score,RFM_Score,Segment
0,2001,4,1,1,6,At Risk
1,2004,4,1,4,9,Potential
2,2005,3,4,3,10,Loyal
3,2006,1,1,1,3,Lost
4,2007,5,1,1,7,Potential
5,2008,4,4,4,12,Loyal
6,2009,2,3,5,10,Loyal
7,2012,2,3,3,8,Potential
8,2013,3,1,2,6,At Risk
9,2014,1,4,4,9,Potential


#### Segmentide kokkuvõte

In [64]:
segment_summary = (
    rfm_scored
    .groupby(
        "Segment",
        observed=True
    )
    .agg(
        customers=("customer_id", "nunique"),
        average_recency=("recency", "mean"),
        average_frequency=("frequency", "mean"),
        average_monetary=("monetary", "mean"),
        total_monetary=("monetary", "sum")
    )
    .reset_index()
)

segment_summary["customer_share_pct"] = (
    segment_summary["customers"]
    / rfm_scored["customer_id"].nunique()
    * 100
)

segment_summary["monetary_share_pct"] = (
    segment_summary["total_monetary"]
    / rfm_scored["monetary"].sum()
    * 100
)

#### Segmentide loogiline järjestus

In [65]:
segment_order = [
    "VIP Champions",
    "Loyal",
    "Potential",
    "At Risk",
    "Lost"
]

segment_summary["Segment"] = pd.Categorical(
    segment_summary["Segment"],
    categories=segment_order,
    ordered=True
)

segment_summary = (
    segment_summary
    .sort_values("Segment")
    .reset_index(drop=True)
)

segment_summary

,Segment,customers,average_recency,average_frequency,average_monetary,total_monetary,customer_share_pct,monetary_share_pct
0,VIP Champions,455,534.661538,7.679121,2519.330000,1146295.15,17.913386,42.822531
1,Loyal,679,631.294551,3.842415,1172.838262,796357.18,26.732283,29.749781
2,Potential,759,693.486166,2.486166,687.474150,521792.88,29.881890,19.492791
3,At Risk,529,795.568998,1.589792,363.270737,192170.22,20.826772,7.178967
4,Lost,118,1002.881356,1.008475,171.483983,20235.11,4.645669,0.755930


#### Segmentide kvaliteedikontroll

In [66]:
print(
    "Segmendita kliendid:",
    rfm_scored["Segment"].isna().sum()
)

print(
    "Kliente segmentides kokku:",
    segment_summary["customers"].sum()
)

print(
    "Kliente RFM-tabelis:",
    rfm_scored["customer_id"].nunique()
)

print(
    "Klientide osakaal kokku:",
    round(segment_summary["customer_share_pct"].sum(), 2)
)

print(
    "Monetary osakaal kokku:",
    round(segment_summary["monetary_share_pct"].sum(), 2)
)

Segmendita kliendid: 0
Kliente segmentides kokku: 2540
Kliente RFM-tabelis: 2540
Klientide osakaal kokku: 100.0
Monetary osakaal kokku: 100.0


#### RFM-skooride ja segmentide tulemused

- Kõik R-, F- ja M-skoorid jäävad vahemikku 1–5.
- RFM-koondskoori vahemik on 3–15.
- Kõik 2 540 klienti said RFM-segmendi.
- Suurim segment on `Potential`, kuhu kuulub 759 klienti ehk
  29,88% analüüsitud klientidest.
- `VIP Champions` segmenti kuulub 455 klienti ehk 17,91%
  kliendibaasist, kuid nende tehingud moodustavad 42,82% kogu
  RFM-andmestiku müügisummast.
- `Loyal` segmenti kuulub 679 klienti ning nende tehingud
  moodustavad 29,75% müügisummast.
- `At Risk` ja `Lost` segmentides on kokku 647 klienti, kuid nende
  tehingud moodustavad suhteliselt väikese osa kogusummast.
- Segmentide kliendi- ja Monetary osakaalud annavad kokku 100%.

## 9. RFM-tulemuste sisuline kontroll

Kontrollin, kas segmentide keskmised RFM-näitajad vastavad
segmenteerimise loogikale.

Segmentide järjestuses `VIP Champions` kuni `Lost` peaks:

- keskmine Recency suurenema;
- keskmine Frequency vähenema;
- keskmine Monetary vähenema.

Samuti arvutan, kui suur osa klientidest ja müügisummast koondub
kõrgema väärtusega ning riskisegmentidesse.

### 9.1. Segmentide koondtabel

In [67]:
segment_summary_display = segment_summary.copy()

segment_summary_display[
    [
        "average_recency",
        "average_frequency",
        "average_monetary",
        "total_monetary",
        "customer_share_pct",
        "monetary_share_pct"
    ]
] = (
    segment_summary_display[
        [
            "average_recency",
            "average_frequency",
            "average_monetary",
            "total_monetary",
            "customer_share_pct",
            "monetary_share_pct"
        ]
    ]
    .round(2)
)

segment_summary_display

,Segment,customers,average_recency,average_frequency,average_monetary,total_monetary,customer_share_pct,monetary_share_pct
0,VIP Champions,455,534.66,7.68,2519.33,1146295.15,17.91,42.82
1,Loyal,679,631.29,3.84,1172.84,796357.18,26.73,29.75
2,Potential,759,693.49,2.49,687.47,521792.88,29.88,19.49
3,At Risk,529,795.57,1.59,363.27,192170.22,20.83,7.18
4,Lost,118,1002.88,1.01,171.48,20235.11,4.65,0.76


### 9.2. Segmentide järjestuse kontroll

Kontrollin segmentide keskmiste näitajate põhjal, kas kliendikäitumine
muutub `VIP Champions` segmendist `Lost` segmendi suunas ootuspäraselt.

In [68]:
segment_logic_check = (
    segment_summary
    .set_index("Segment")
)

print(
    "Keskmine Recency suureneb VIP-ist Lost-segmendi suunas:",
    segment_logic_check["average_recency"].is_monotonic_increasing
)

print(
    "Keskmine Frequency väheneb VIP-ist Lost-segmendi suunas:",
    segment_logic_check["average_frequency"].is_monotonic_decreasing
)

print(
    "Keskmine Monetary väheneb VIP-ist Lost-segmendi suunas:",
    segment_logic_check["average_monetary"].is_monotonic_decreasing
)

Keskmine Recency suureneb VIP-ist Lost-segmendi suunas: True
Keskmine Frequency väheneb VIP-ist Lost-segmendi suunas: True
Keskmine Monetary väheneb VIP-ist Lost-segmendi suunas: True


### 9.3. Prioriteetsete kliendirühmade osakaalud

Koondan `VIP Champions` ja `Loyal` segmendid kõrgema väärtusega
kliendirühmaks ning `At Risk` ja `Lost` segmendid riskirühmaks.

In [69]:
high_value_segments = segment_summary[
    segment_summary["Segment"].isin(
        ["VIP Champions", "Loyal"]
    )
]

risk_segments = segment_summary[
    segment_summary["Segment"].isin(
        ["At Risk", "Lost"]
    )
]

print(
    "VIP ja Loyal kliente kokku:",
    high_value_segments["customers"].sum()
)

print(
    "VIP ja Loyal klientide osakaal:",
    round(
        high_value_segments["customer_share_pct"].sum(),
        2
    ),
    "%"
)

print(
    "VIP ja Loyal Monetary osakaal:",
    round(
        high_value_segments["monetary_share_pct"].sum(),
        2
    ),
    "%"
)

print(
    "\nAt Risk ja Lost kliente kokku:",
    risk_segments["customers"].sum()
)

print(
    "At Risk ja Lost klientide osakaal:",
    round(
        risk_segments["customer_share_pct"].sum(),
        2
    ),
    "%"
)

print(
    "At Risk ja Lost Monetary osakaal:",
    round(
        risk_segments["monetary_share_pct"].sum(),
        2
    ),
    "%"
)

VIP ja Loyal kliente kokku: 1134
VIP ja Loyal klientide osakaal: 44.65 %
VIP ja Loyal Monetary osakaal: 72.57 %

At Risk ja Lost kliente kokku: 647
At Risk ja Lost klientide osakaal: 25.47 %
At Risk ja Lost Monetary osakaal: 7.93 %


### 9.4. Kõrgeima väärtusega kliendid

Järjestan kliendid esmalt RFM-koondskoori ja seejärel Monetary,
Frequency ning Recency järgi.

Avalikus notebook’is kuvan kliendi tuvastamiseks ainult
`customer_id`. Nime ja e-posti ei kuvata.

In [70]:
top_customers = (
    rfm_scored
    .sort_values(
        by=[
            "RFM_Score",
            "monetary",
            "frequency",
            "recency"
        ],
        ascending=[
            False,
            False,
            False,
            True
        ]
    )
    [
        [
            "customer_id",
            "Segment",
            "RFM_Score",
            "R_score",
            "F_score",
            "M_score",
            "recency",
            "frequency",
            "monetary",
            "last_purchase_date"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

top_customers

,customer_id,Segment,RFM_Score,R_score,F_score,M_score,recency,frequency,monetary,last_purchase_date
0,3618,VIP Champions,15,5,5,5,493,72,27920.86,2025-02-21
1,3350,VIP Champions,15,5,5,5,505,76,26286.10,2025-02-09
2,2889,VIP Champions,15,5,5,5,20,71,24172.58,2026-06-09
3,2997,VIP Champions,15,5,5,5,179,77,23667.93,2026-01-01
4,3648,VIP Champions,15,5,5,5,494,72,22942.42,2025-02-20
5,4206,VIP Champions,15,5,5,5,486,69,21916.75,2025-02-28
6,3605,VIP Champions,15,5,5,5,490,77,21626.10,2025-02-24
7,2221,VIP Champions,15,5,5,5,197,65,21610.39,2025-12-14
8,2154,VIP Champions,15,5,5,5,496,66,20726.79,2025-02-18
9,3722,VIP Champions,15,5,5,5,514,62,20317.81,2025-01-31


#### TOP-klientide tõlgendus

- Kõik TOP 10 kliendid kuuluvad segmenti `VIP Champions` ja said
  maksimaalse RFM-koondskoori 15.
- Nende Frequency jääb vahemikku 62–77 tehingut ning Monetary
  ligikaudu 20 318–27 921 eurot.
- Kõrge R-skoor on suhteline: see näitab kuulumist kliendibaasi
  kõige hiljutisema ostukäitumisega kvintiili.
- Mitme VIP-kliendi viimasest ostust on möödunud ligikaudu
  486–514 päeva. Seetõttu ei tohi segmenti `VIP Champions`
  automaatselt tõlgendada kui praegu aktiivsete klientide rühma.
- Segmentide ärilisel kasutamisel tuleb RFM-skoori kõrval vaadata
  ka Recency tegelikku päevade arvu.

#### Kontrollin Recency skoori tegelikke piire

In [71]:
recency_score_ranges = (
    rfm_scored
    .groupby("R_score")
    .agg(
        customers=("customer_id", "count"),
        min_recency=("recency", "min"),
        max_recency=("recency", "max"),
        average_recency=("recency", "mean")
    )
    .sort_index(ascending=False)
)

recency_score_ranges["average_recency"] = (
    recency_score_ranges["average_recency"]
    .round(2)
)

recency_score_ranges

,customers,min_recency,max_recency,average_recency
R_score,,,,
5,511,1,545,495.27
4,508,546,605,571.25
3,507,606,689,647.53
2,507,690,811,743.68
1,507,812,1269,964.20


#### Recency skooride tõlgendus

- Recency skoorid jaotusid viide ligikaudu võrdse suurusega
  kliendirühma.
- Kõrgeima `R_score = 5` saanud klientide viimasest ostust on
  möödunud 1–545 päeva.
- Madalaima `R_score = 1` saanud klientide viimasest ostust on
  möödunud 812–1 269 päeva.
- R-skoor väljendab kliendi positsiooni ülejäänud kliendibaasi
  suhtes, mitte absoluutset kliendi aktiivsust.
- Seetõttu võib ka enam kui aasta tagasi ostnud klient saada
  kõrgeima R-skoori, kui ülejäänud kliendibaasi ostud on veel
  varasemad.
- RFM-segmentide ärilisel tõlgendamisel tuleb lisaks skoorile
  arvestada Recency tegelikku päevade arvu.

## 10. Visualiseerimine

Visualiseerin RFM-segmentide suuruse ja rahalise väärtuse.

Esimene diagramm näitab klientide arvu igas segmendis. Teine diagramm
näitab segmentide osakaalu kogu RFM-andmestiku Monetary väärtusest.

### 10.1. Klientide arv segmentide kaupa

In [72]:
fig_customers = px.bar(
    segment_summary_display,
    x="Segment",
    y="customers",
    text="customers",
    category_orders={
        "Segment": [
            "VIP Champions",
            "Loyal",
            "Potential",
            "At Risk",
            "Lost"
        ]
    },
    title="Klientide arv RFM-segmentide kaupa"
)

fig_customers.update_traces(
    textposition="outside"
)

fig_customers.update_layout(
    xaxis_title="RFM-segment",
    yaxis_title="Klientide arv",
    showlegend=False
)

fig_customers.show()

### 10.2. Segmentide osakaal kogu Monetary väärtusest

Diagramm näitab, kui suur osa analüüsitud klientide positiivsete
tehingute kogusummast pärineb igast RFM-segmendist.

In [73]:
fig_monetary_share = px.bar(
    segment_summary_display,
    x="Segment",
    y="monetary_share_pct",
    text="monetary_share_pct",
    category_orders={
        "Segment": [
            "VIP Champions",
            "Loyal",
            "Potential",
            "At Risk",
            "Lost"
        ]
    },
    title="RFM-segmentide osakaal kogu Monetary väärtusest"
)

fig_monetary_share.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig_monetary_share.update_layout(
    xaxis_title="RFM-segment",
    yaxis_title="Monetary osakaal (%)",
    showlegend=False
)

fig_monetary_share.show()

### 10.3. Segmentide keskmised RFM-näitajad

Kuna Recency, Frequency ja Monetary on erineva mõõtkavaga, kuvan
need eraldi diagrammidel, mitte ühel ühisel teljel.

#### Keskmine Recency

In [74]:
fig_recency = px.bar(
    segment_summary_display,
    x="Segment",
    y="average_recency",
    text="average_recency",
    category_orders={"Segment": segment_order},
    title="Keskmine Recency segmentide kaupa"
)

fig_recency.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)

fig_recency.update_layout(
    xaxis_title="RFM-segment",
    yaxis_title="Päevi viimasest ostust",
    showlegend=False
)

fig_recency.show()

#### Keskmine Frequency

In [75]:
fig_frequency = px.bar(
    segment_summary_display,
    x="Segment",
    y="average_frequency",
    text="average_frequency",
    category_orders={"Segment": segment_order},
    title="Keskmine Frequency segmentide kaupa"
)

fig_frequency.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig_frequency.update_layout(
    xaxis_title="RFM-segment",
    yaxis_title="Keskmine tehingute arv",
    showlegend=False
)

fig_frequency.show()

#### Keskmine Monetary

In [76]:
fig_monetary = px.bar(
    segment_summary_display,
    x="Segment",
    y="average_monetary",
    text="average_monetary",
    category_orders={"Segment": segment_order},
    title="Keskmine Monetary segmentide kaupa"
)

fig_monetary.update_traces(
    texttemplate="%{text:.2f} €",
    textposition="outside"
)

fig_monetary.update_layout(
    xaxis_title="RFM-segment",
    yaxis_title="Keskmine müügisumma (€)",
    showlegend=False
)

fig_monetary.show()

## 11. Ärilised järeldused ja soovitused

RFM-analüüs jagas 2 540 klienti viide ostukäitumisel põhinevasse
segmenti.

### Peamised tulemused

- `VIP Champions` segmenti kuulub 455 klienti ehk 17,91% analüüsitud
  klientidest. Nende tehingud moodustavad 42,82% kogu RFM-andmestiku
  Monetary väärtusest.
- `Loyal` segmenti kuulub 679 klienti ehk 26,73% klientidest ning
  nende Monetary osakaal on 29,75%.
- `VIP Champions` ja `Loyal` moodustavad kokku 44,65% klientidest,
  kuid 72,57% kogu Monetary väärtusest.
- Suurim segment on `Potential`, kuhu kuulub 759 klienti ehk 29,88%
  kliendibaasist. Nende Monetary osakaal on 19,49%.
- `At Risk` ja `Lost` segmentides on kokku 647 klienti ehk 25,47%
  analüüsitud klientidest, kuid nende Monetary osakaal on ainult
  7,93%.
- Kõrge väärtusega kliendid eristuvad eelkõige suurema ostusageduse
  ja kogukulutuse poolest. TOP 10 kliendid tegid 62–77 tehingut ning
  nende Monetary jäi ligikaudu 20 318–27 921 euro vahele.

### Soovitused segmentide kaupa

| Segment | Äriline tõlgendus | Soovitatud tegevus |
|---|---|---|
| `VIP Champions` | Suurima suhtelise väärtusega kliendid, kellel on kõrged R-, F- ja M-skoorid | Hoida kliendisuhet, pakkuda personaalseid hüvesid ja varajast ligipääsu. Tegeliku aktiivsuse hindamisel vaadata lisaks Recency päevade arvu. |
| `Loyal` | Sagedased ja väärtuslikud kliendid, kuid näitajad jäävad VIP-rühmast madalamaks | Suurendada kordusoste, kasutada lojaalsusprogramme ning sobivate toodete ristmüüki. |
| `Potential` | Suurim segment, mille ostukäitumine on keskmisel tasemel | Suunata järgmisele ostule, testida personaalseid soovitusi ja ajastatud kordusostu pakkumisi. |
| `At Risk` | Suhteliselt vanema viimase ostu ning väiksema sageduse ja väärtusega kliendid | Rakendada sihitud taasaktiveerimist. Enne soodustuse pakkumist arvestada kliendi varasema väärtusega. |
| `Lost` | Kõige madalama RFM-koondskooriga kliendid | Kasutada madala kuluga taastamiskampaaniaid või jätta aktiivsest turundusest välja, kui reageerimist ei toimu. |

### Recency mõju otsustele

RFM-skoorid on suhtelised ja võrdlevad klienti ülejäänud
kliendibaasiga.

Kõrgeim `R_score = 5` hõlmab selles analüüsis kliente, kelle viimasest
ostust on möödunud 1–545 päeva. Seetõttu ei tähenda kõrge R-skoor
automaatselt, et klient on praegu aktiivne.

Turundustegevuste lõplikul valikul tuleb kasutada koos:

- RFM-segmenti;
- Recency tegelikku päevade arvu;
- kliendi kontaktandmete olemasolu;
- ettevõtte määratud aktiivsus- ja kadumispiire.

## 12. Piirangud ja eeldused

Analüüsi tulemuste tõlgendamisel tuleb arvestada järgmiste
piirangutega.

### Andmete ulatus ja puhastamine

- Analüüs põhineb UrbanStyle’i koolitusandmestikul.
- Ühendatud lähteandmestikus oli 10 118 müügirida.
- RFM-analüüsist jäeti välja 988 puuduva `customer_id`-ga rida,
  sest neid ei olnud võimalik siduda konkreetse kliendiga.
- Null- või negatiivse müügisummaga ridu oli 195. Need jäeti
  koolituse RFM-metoodika järgi Monetary arvutusest välja.
- Välistamise põhjused kattusid 15 real ning kokku eemaldati
  1 168 unikaalset rida.
- Lõplik RFM-analüüs põhineb 8 950 positiivse summaga tehingul ja
  2 540 kliendil.

### RFM-metoodika eeldused

- Recency arvutamise viitekuupäev on 2026-06-29 ehk üks päev pärast
  andmestiku viimast müügikuupäeva.
- Frequency põhineb unikaalsete `sale_id` väärtuste arvul.
- Monetary põhineb positiivsete `total_price` väärtuste summal ning
  ei kajasta toodete omahinda, marginaali ega kliendi kasumlikkust.
- R-, F- ja M-skoorid määrati kvintiilide alusel. Need väljendavad
  kliendi suhtelist positsiooni selles andmestikus.
- Segmendipiirid põhinevad koolitusjuhendi RFM-koondskoori
  vahemikel ega ole valideeritud kliendi hilisema tegeliku käitumise
  või ettevõtte kampaaniatulemustega.

### Ärilise kasutamise piirangud

- Analüüs ei arvesta toodete, kategooriate, müügikanalite,
  kampaaniate ega hooajalisuse mõju.
- Sama e-posti aadressiga kliendiridu oli 258 ning e-post puudus
  380 kliendil. See ei mõjutanud `customer_id` põhist RFM-arvutust,
  kuid võib mõjutada turundussegmentide rakendamist.
- `loyalty_tier` puudus 1 260 kliendil, mistõttu RFM-segmente ei
  võrreldud täielikult olemasolevate lojaalsustasemetega.
- Enne segmentide kasutamist päris turundustegevustes tuleb määrata
  ettevõtte ostutsükliga sobivad absoluutsed aktiivsus-,
  passiivsus- ja kadumispiirid.